## HyDE (Hypothetical Document Embeddings，假设性文档嵌入)
🧠 What is HyDe?

HyDE (Hypothetical Document Embeddings) is a retrieval technique where, instead of embedding the user’s query directly, you first generate a hypothetical answer (document) to the query using an LLM — and then embed that hypothetical document to search your vector store.

➡️ HyDE bridges the gap between user intent and relevant content, especially when:

1. Queries are short
2. Language mismatch between query and documents
3.You want to retrieve based on answer content, not question words

In [8]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [4]:
# 1. Load and chunk your dataset
chunk_size = 300
chunk_overlap = 100

# loading data
loader = WikipediaLoader(query = "Steve Jobs", load_max_docs=5)
documents = loader.load()

# chunking documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
chunks = text_splitter.split_documents(documents)
chunks

d:\codes\agent_study\RAGUdemy\.venv\Lib\site-packages\wikipedia\wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file d:\codes\agent_study\RAGUdemy\.venv\Lib\site-packages\wikipedia\wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


[Document(metadata={'title': 'Steve Jobs', 'summary': 'Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, co-inventor, and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Inc. (as Apple Computer Company) with Steve Wozniak and Ronald Wayne in 1976. After the company\'s board of directors fired him in 1985, he founded NeXT the same year and purchased Pixar in 1986, becoming its chairman and majority shareholder until 2007. Jobs returned to Apple in 1997 as CEO, where he was closely involved with the creation and promotion of many of the company\'s most influential products until his resignation in 2011.\nJobs was born in San Francisco in 1955 and adopted shortly afterwards. He attended Reed College in 1972 before withdrawing that same year. In 1974, he traveled through India, seeking enlightenment before later studying Zen Buddhism. He and Wozniak co-founded Apple in 1976 to further develop and sell 

In [5]:
# Step 2: Vector Store
embedding_model = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embedding_model)

# Step 3: MMR Retriever
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 5, "lambda_mult": 0.7})
retriever

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000021C19E392B0>, search_type='mmr', search_kwargs={'k': 5, 'lambda_mult': 0.7})

In [7]:
# Step 4: LLM and Prompt

import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.4,
    api_key=api_key,
    base_url=base_url
)
llm

ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000021C1DCCAE40>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000021C1DCCBA10>, root_client=<openai.OpenAI object at 0x0000021C1DCC82F0>, root_async_client=<openai.AsyncOpenAI object at 0x0000021C1DCCB770>, temperature=0.4, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://yibuapi.com/v1')

In [14]:
from langchain_classic.prompts.chat import SystemMessagePromptTemplate, ChatPromptTemplate

def get_hyde_doc(query):
    template = """Imagine you are an expert writing a detailed explanation on the topic: '{query}'
    create a hypothetical answer for the topic"""

    system_message_prompt = SystemMessagePromptTemplate.from_template(template)
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt])
    messages = chat_prompt.format_prompt(query=query).to_messages()

    # print(f"system message prompt: {system_message_prompt}")
    # print(f"chat prompt: {chat_prompt}")
    print(f"messages: {messages}")

    response = llm.invoke(messages)
    hypo_doc = response.content
    return hypo_doc

In [15]:
query = "When was Steve Jobs fired from Apple?"
print(get_hyde_doc(query))

messages: [SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'\n    create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
Steve Jobs was effectively forced out of Apple in 1985. After co-founding the company in 1976 and leading it to early success with products like the Apple II and the original Macintosh, internal power struggles and differing visions for the company's future led to tensions between Jobs and then-CEO John Sculley, whom Jobs had recruited from PepsiCo.

By mid-1985, the Apple board sided with Sculley during a power struggle, resulting in Jobs being stripped of his managerial duties. Feeling marginalized, Jobs resigned from Apple later that year. This departure marked a significant turning point in his career, after which he founded NeXT and acquired Pixar before eventually returning to Apple in 1997 to lead its remarkable resurgence.


In [16]:
matched_doc = retriever.invoke(get_hyde_doc(query))
print(matched_doc)

messages: [SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'When was Steve Jobs fired from Apple?'\n    create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
[Document(id='8ec2a2ae-60ec-4a63-bd72-3396acc6fa92', metadata={'title': 'Steve Jobs', 'summary': 'Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, co-inventor, and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs co-founded Apple Inc. (as Apple Computer Company) with Steve Wozniak and Ronald Wayne in 1976. After the company\'s board of directors fired him in 1985, he founded NeXT the same year and purchased Pixar in 1986, becoming its chairman and majority shareholder until 2007. Jobs returned to Apple in 1997 as CEO, where he was closely involved with the creation and promotion of many of the company\'s most influential products until his resignation in 2011.\nJobs was born in S

# Langchain - HypotheticalDocumentEmbedder

In [17]:
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder

from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# step 1: load and split documents
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

In [20]:
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm = llm,
    base_embeddings = embedding_model,
    prompt_key = "web_search"
)
hyde_embedding_function

HypotheticalDocumentEmbedder(verbose=False, base_embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), llm_chain=PromptTemplate(input_variables=['QUESTION'], input_types={}, partial_variables={}, template='Please write a passage to answer the question\nQuestion: {QUESTION}\nPassage:')
| ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000021C1DCCAE40>, 

According to the official documentation and LangChain source code (mapping in PROMPT_MAP), the default options are:

- web_search
- sci_fact
- arguana
- trec_covid
- fiqa
- dbpedia_entity
- trec_news
- mr_tydi

![image.png](attachment:image.png)

In [22]:
vector_store = Chroma.from_documents(
    documents = chunks,
    embedding = hyde_embedding_function,
    persist_directory = "output/langchain"
)

In [23]:
# Step 5: RAG answer generation prompt
rag_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")
rag_chain = create_stuff_documents_chain(llm=llm, prompt=rag_prompt)

In [26]:
# Step 6: Final RAG Pipeline
def hyde_rag_pipeline(query):
    # Retrieve relevant documents
    relevant_documents = vector_store.similarity_search(query, k = 4)
    print(f" {relevant_documents}")
    
    # Generate answer
    answer = rag_chain.invoke({"context": relevant_documents, "input": query})
    
    return answer

In [27]:
# Step 7: Run example query
query = "What memory modules does LangChain provide?"
answer = hyde_rag_pipeline(query)
print("✅ Final Answer:\n", answer)

 [Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v10)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v2)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v5)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_con

## Custom Prompt

In [29]:
from langchain_classic.prompts import PromptTemplate

custom = PromptTemplate.from_template(
    "Generate a concise hypothetical answer to the question: {question}"
)

hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=embedding_model,
    custom_prompt=custom
)